<a href="https://colab.research.google.com/github/rahiakela/small-language-models-fine-tuning/blob/main/domain-specific-small-language-models/02-running-inference/03_end_to_end_small_language_model_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## End-to-End Small Language Model Fine Tuning to Generate Manim Code

We’ll fine-tune the GPT-2 small model (https://huggingface.co/
openai-community/gpt2) to generate working Manim code from natural-language
prompts. Manim (https://github.com/ManimCommunity/manim) is an open
source Python animation engine for explanatory math videos—it creates precise
animations programmatically.

Hardware acceleration (GPU) is needed.

Install the missing dependencies in the Colab VM (only Optuna for hyperparameter search isn't available by default).

In [ ]:
!pip install optuna

## Data Preparation

Download the [Manim_Python dataset](https://huggingface.co/datasets/Edoh/manim_python) from the Hugging Face's Hub.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Edoh/manim_python")

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 599
    })
    test: Dataset({
        features: ['instruction', 'output'],
        num_rows: 51
    })
})

Display some training sample.

In [4]:
print(dataset["train"][0])

{'instruction': "Create a new scene named 'MyScene'.", 'output': 'from manim import * class MyScene(Scene): def construct(self): pass'}


Download the GPT2 Small model's tokenizer.

In [ ]:
from transformers import GPT2Tokenizer

model_name = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

Set pad token to eos token for the model's tokenizer.

In [6]:
tokenizer.pad_token = tokenizer.eos_token

Define a custom function to preprocess the raw data. It concatenates the `instruction` and `output` columms of the dataset, tokenizes the concatenated text and set the lables same as the input_ids.

In [7]:
def preprocess_data(examples):
    inputs = [
        f"Instruction: {instr}\nOutput: {out}"
        for instr, out in zip(examples["instruction"], examples["output"])
    ]
    tokenized = tokenizer(inputs, truncation=True, max_length=512, padding="max_length")

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

Use the `preprocess_data` function to tokenize the training data. The original columns are also removed, as they aren't needed anymore for training purposes.

In [ ]:
tokenized_datasets = dataset.map(preprocess_data,
                                 batched=True,
                                 remove_columns=dataset["train"].column_names)

The data is now ready, and we can prepare the model for training.

## Hyperparameter Tuning

Import the required classes.

In [9]:
from transformers import (
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

Define a custom function to init the model during the hyperparameter search. This way we will have a fresh model for each trial. The baseline model weights are pulled from the HF's Hub only during the first trial. Subsequents trials will use the cached weights.

In [10]:
def model_init():
    return GPT2LMHeadModel.from_pretrained(model_name, device_map='auto')

Set the training arguments.

In [11]:
training_args = TrainingArguments(
    output_dir="./gpt2-manim-python-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="none",
)

Split the training data, to reserve 10% for validation.

In [12]:
train_val_split = tokenized_datasets["train"].train_test_split(test_size=0.1)
tokenized_datasets["train"] = train_val_split["train"]
tokenized_datasets["validation"] = train_val_split["test"]

Create an instance of data collator for causal language modeling.

In [13]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Create an instance of Trainer. An early stopping callback is set too.

In [ ]:
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Define the hyperparameter search space.

In [16]:
def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [2, 4, 8]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 6),
        "warmup_steps": trial.suggest_int("warmup_steps", 0, 500),
        "gradient_accumulation_steps": trial.suggest_categorical("gradient_accumulation_steps", [1, 2, 4]),
    }

Run the hyperparameter search. The Optuna backend is used.

In [17]:
trials_count = 3
best_run = trainer.hyperparameter_search(
    direction="minimize",
    backend="optuna",
    n_trials=trials_count,
    hp_space=hp_space,
    compute_objective=lambda metrics: metrics["eval_loss"],
)

[I 2026-06-01 05:10:11,698] A new study created in memory with name: no-name-65be157b-4241-4a65-b73b-0453dde41837


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,2.244139
2,No log,0.777897
3,1.957078,0.551010


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
[I 2026-06-01 05:28:15,303] Trial 2 finished with value: 0.24095304310321808 and parameters: {'learning_rate': 2.4900489938010723e-05, 'per_device_train_batch_size': 4, 'weight_decay': 0.055252799060903135, 'num_train_epochs': 6, 'warmup_steps': 197, 'gradient_accumulation_steps': 4}. Best is trial 1 with value: 0.230420783162117.


Display the best combination of hyperparmaters found.

In [18]:
print("Best hyperparameters found:", best_run.hyperparameters)

Best hyperparameters found: {'learning_rate': 0.00043950378826472167, 'per_device_train_batch_size': 4, 'weight_decay': 0.27938091990645164, 'num_train_epochs': 6, 'warmup_steps': 340, 'gradient_accumulation_steps': 2}


## Training Model

In [26]:
# pre-computed hyperparameters values
best_hyperparameters = {
    'learning_rate': 0.00043950378826472167,
    'per_device_train_batch_size': 4,
    'weight_decay': 0.27938091990645164,
    'num_train_epochs': 6,
    'warmup_steps': 340,
    'gradient_accumulation_steps': 2
}

In [27]:
for key, value in best_hyperparameters.items():
    print(key, value)

learning_rate 0.00043950378826472167
per_device_train_batch_size 4
weight_decay 0.27938091990645164
num_train_epochs 6
warmup_steps 340
gradient_accumulation_steps 2


Configure the Trainer with the best hyperparameters.

In [19]:
# for key, value in best_run.hyperparameters.items():
for key, value in best_hyperparameters.items():
    setattr(training_args, key, value)

In [ ]:
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets.get("validation"),
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Start the model fine-tuning.

In [22]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,No log,0.307881
2,1.054280,0.248680
3,0.197792,0.206604
4,0.197792,0.227873
5,0.165602,0.230421


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=340, training_loss=0.4365622618619133, metrics={'train_runtime': 524.6304, 'train_samples_per_second': 6.164, 'train_steps_per_second': 0.778, 'total_flos': 704182026240000.0, 'train_loss': 0.4365622618619133, 'epoch': 5.0})

Save the fine-tuned model and the tokenizer to disk.

In [24]:
trainer.save_model("./gpt2-manim-python-finetuned")
tokenizer.save_pretrained("./gpt2-manim-python-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-manim-python-finetuned/tokenizer_config.json',
 './gpt2-manim-python-finetuned/tokenizer.json')

## Testing Model

Load the fine-tuned model from disk to GPU memory. Load also its companion tokenizer.

In [28]:
import torch

model_dir = "./gpt2-manim-python-finetuned"
tokenizer = GPT2Tokenizer.from_pretrained(model_dir)
model = GPT2LMHeadModel.from_pretrained(model_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

Define a custom function to do Manim code generation with the fine-tuned model and decode the output.

In [29]:
def generate_output(instruction, max_length=150, num_beams=5, temperature=0.7, top_p=0.9, repetition_penalty=1.2):
    """
    Generate output text given an instruction using beam search and nucleus sampling.

    Args:
        instruction (str): The input instruction prompt.
        max_length (int): Maximum length of generated sequence (including prompt).
        num_beams (int): Number of beams for beam search.
        temperature (float): Sampling temperature; lower is less random.
        top_p (float): Nucleus sampling probability threshold.
        repetition_penalty (float): Penalty for repeated tokens (>1.0 discourages repetition).

    Returns:
        str: Generated output text.
    """

    prompt = f"Instruction: {instruction}\nOutput:"

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    generated_ids = model.generate(
        input_ids,
        max_length=max_length,
        num_beams=num_beams,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        early_stopping=True,
        no_repeat_ngram_size=2,
    )

    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    output_start = generated_text.find("Output:")
    if output_start != -1:
        output_text = generated_text[output_start + len("Output:"):].strip()
    else:
        output_text = generated_text.strip()

    return output_text

Execute the fine-tuned model on all the samples in the test set and save the generated Manim code snippets (along with the corresponding prompt and ground truth) in a CSV file.

In [30]:
import csv

output_csv = "gpt2_manim_python_test_outputs.csv"
with open(output_csv, mode="w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=["instruction", "reference_output", "generated_output"])
    writer.writeheader()

    for example in dataset['test']:
        instruction = example["instruction"]
        reference_output = example["output"]

        generated_output = generate_output(instruction)

        writer.writerow({
            "instruction": instruction,
            "reference_output": reference_output,
            "generated_output": generated_output,
        })

print(f"Inference complete. Results saved to {output_csv}")

Inference complete. Results saved to gpt2_manim_python_test_outputs.csv


In [33]:
import pandas as pd

df = pd.read_csv(output_csv)
df.head()

,instruction,reference_output,generated_output
0,Rotate the pentagon by 90 degrees counterclock...,from manim import * class MyScene(Scene): def ...,from manim import * class MyScene(Scene): def ...
1,"Create a rectangle with width 4 and height 2, ...",from manim import * class MyScene(Scene): def ...,from manim import * class MyScene(Scene): def ...
2,Create a VGroup and add a circle and a triangl...,from manim import * class MyScene(Scene): def ...,from manim import * class MyScene(Scene): def ...
3,Rotate the VGroup by 45 degrees clockwise over...,from manim import * class MyScene(Scene): def ...,from manim import * class MyScene(Scene): def ...
4,Create a regular hexagon with side length 2 an...,from manim import * class MyScene(Scene): def ...,from manim import * class MyScene(Scene): def ...
